# 10 — External Data + Expanded Multitask Chemprop

Expands the training signal by pulling data from ChEMBL and PubChem for PXR and
structurally-related nuclear receptors, then trains a wider multitask Chemprop model.

**Task matrix** (each column is an output head; NaN = not measured for that compound):

| Task | Source | Scale | # compounds (est.) |
|---|---|---|---|
| pec50_pxr | Challenge train | pEC50 | 3,781 (cleaned) |
| pec50_null | Counter assay | pEC50 | 2,649 |
| log2fc_sp | Single-conc screen | log2FC | 21,003 |
| chembl_pxr | ChEMBL CHEMBL3401 | pEC50 | ~3,000 |
| chembl_car | ChEMBL CHEMBL3509594 | pEC50 | ~500 |
| chembl_vdr | ChEMBL CHEMBL1977 | pEC50 | ~2,000 |
| chembl_fxr | ChEMBL CHEMBL2047 | pEC50 | ~2,000 |
| chembl_pparg | ChEMBL CHEMBL235 | pEC50 | ~10,000 |
| logP | RDKit (computed) | continuous | all |
| MW | RDKit (computed) | continuous | all |
| TPSA | RDKit (computed) | continuous | all |

**Design rationale**:
- Nuclear receptor tasks share a binding-pocket vocabulary → shared encoder benefits
- Single-conc log2FC gives 21k additional structure–activity pairs for PXR directly
- Chemical property tasks (logP / MW / TPSA) have labels for *every* compound,
  giving the encoder a dense signal even for external SMILES with no bioassay data
- Pseudo-inactive SMILES from `09_data_pipeline` pad the inactive region

**Expected runtime**: ~2–3 h on CPU (data fetch + 5-fold CV for validation, then final model)

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings, time
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as L
import chemprop
from chemprop import data as cdata, models as cmodels, nn as cnn
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

from pxr import data as D
from pxr.chem import to_inchikey, standardize, bemis_murcko
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, FIGURES, SUBMISSIONS
from pxr.eval import compute_metrics, scaffold_kfold_indices
from pxr.external import (
    NUCLEAR_RECEPTOR_TARGETS, PUBCHEM_PXR_AIDS,
    fetch_all_nr_targets, fetch_pubchem_aid, standardize_external,
)
from pxr.preprocess import pseudo_inactive_augment

plt.rcParams.update({"figure.dpi": 120})
print(f"torch {torch.__version__} | chemprop {chemprop.__version__}")

In [ ]:
# ── 2. Load cleaned training + counter data ───────────────────────────────────
train = pd.read_parquet(DATA_PROCESSED / 'train_clean.parquet')
counter = D.load_counter()
sc = D.load_single_conc()
te = D.load_test()

train['inchikey'] = train['smiles'].map(to_inchikey)
counter['inchikey'] = counter['smiles'].map(to_inchikey)
sc['inchikey'] = sc['smiles'].map(to_inchikey)

train_iks = set(train['inchikey'].dropna())
print(f"Cleaned training compounds: {len(train):,}")
print(f"Counter-assay compounds:    {len(counter):,}")
print(f"Single-conc compounds:      {len(sc):,}")

In [ ]:
# ── 3. Fetch ChEMBL data for all NR targets ───────────────────────────────────
# NOTE: this cell fetches from the internet (~5–15 min for all targets).
# Results are cached to DATA_EXTERNAL / 'chembl_nr_targets.parquet'.

chembl_cache = DATA_EXTERNAL / 'chembl_nr_targets.parquet'

if chembl_cache.exists():
    print("Loading from cache ...")
    chembl_all = pd.read_parquet(chembl_cache)
else:
    print("Fetching from ChEMBL (first run — may take 10–20 min) ...")
    chembl_all = fetch_all_nr_targets()
    chembl_all.to_parquet(chembl_cache, index=False)

print(f"\nChEMBL records by target:")
print(chembl_all.groupby('target_name').size().sort_values(ascending=False).to_string())
print(f"\nTotal unique ChEMBL SMILES: {chembl_all['smiles'].nunique():,}")

In [ ]:
# ── 4. Fetch PubChem qHTS PXR data ────────────────────────────────────────────
# Supplement ChEMBL PXR data with Tox21/NCATS qHTS assays.

pubchem_cache = DATA_EXTERNAL / 'pubchem_pxr_qhts.parquet'

if pubchem_cache.exists():
    print("Loading PubChem from cache ...")
    pubchem_df = pd.read_parquet(pubchem_cache)
else:
    from pxr.external import fetch_pubchem_aid
    dfs = []
    for aid in PUBCHEM_PXR_AIDS:
        print(f"Fetching AID {aid} ...")
        df = fetch_pubchem_aid(aid)
        if not df.empty:
            dfs.append(df)
    pubchem_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    if not pubchem_df.empty:
        pubchem_df.to_parquet(pubchem_cache, index=False)

if not pubchem_df.empty:
    print(f"PubChem qHTS: {len(pubchem_df):,} records")
    print(pubchem_df['activity_outcome'].value_counts())
else:
    print("No PubChem data fetched — proceeding without it.")

In [ ]:
# ── 5. Standardise and deduplicate external data against training set ──────────
chembl_std = standardize_external(chembl_all, train_inchikeys=train_iks)

# Pivot: one column per NR target, rows = unique compounds
chembl_pivot = (
    chembl_std
    .groupby(['inchikey', 'target_name'])
    .agg(pec50=('pec50', 'median'), std_smiles=('std_smiles', 'first'))
    .reset_index()
    .pivot(index='inchikey', columns='target_name', values='pec50')
    .reset_index()
)
# Recover SMILES from the std column (join back)
smi_map = chembl_std.groupby('inchikey')['std_smiles'].first()
chembl_pivot['smiles'] = chembl_pivot['inchikey'].map(smi_map)

print(f"External compounds after dedup: {len(chembl_pivot):,}")
print("Columns:", chembl_pivot.columns.tolist())

In [ ]:
# ── 6. Chemical property auxiliary targets (computed for ALL compounds) ────────
def compute_chem_props(smiles_list: list[str]) -> pd.DataFrame:
    """Compute logP, MW, TPSA for a list of SMILES."""
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol is None:
            rows.append({'logP': np.nan, 'MW': np.nan, 'TPSA': np.nan})
        else:
            rows.append({
                'logP': Descriptors.MolLogP(mol),
                'MW':   Descriptors.MolWt(mol),
                'TPSA': rdMolDescriptors.CalcTPSA(mol),
            })
    return pd.DataFrame(rows)

print("Computing chemical properties for training set ...")
props_train = compute_chem_props(train['smiles'].tolist())
print(props_train.describe().round(2))

In [ ]:
# ── 7. Assemble the full multitask training matrix ────────────────────────────
# Strategy:
#   A) Start with all unique SMILES across all sources
#   B) For each task, fill in labels where available; NaN otherwise
#   C) Chemprop NaN-masks each task in the loss

# --- 7a. Core training set (challenge data)
# Join train + counter by InChIKey
counter_ik = counter.set_index('inchikey')[['pec50']].rename(columns={'pec50': 'pec50_null'})
core = train.join(counter_ik, on='inchikey', how='left')

# Join single-conc by InChIKey (take median log2FC per compound)
sc_agg = sc.groupby('inchikey')['log2_fc_estimate'].median().rename('log2fc_sp')
core = core.join(sc_agg, on='inchikey', how='left')

# Add chemical properties
core = pd.concat([core.reset_index(drop=True), props_train.reset_index(drop=True)], axis=1)

print(f"Core rows (challenge train): {len(core):,}")
for col in ['pec50', 'pec50_null', 'log2fc_sp', 'logP', 'MW', 'TPSA']:
    if col in core.columns:
        pct = core[col].notna().mean()
        print(f"  {col:15s}: {pct:.1%} filled")

In [ ]:
# --- 7b. Add single-conc ONLY compounds (not in train) as extra rows
sc_novel_iks = set(sc['inchikey'].dropna()) - train_iks
sc_novel = (
    sc[sc['inchikey'].isin(sc_novel_iks)]
    .groupby('inchikey')
    .agg(smiles=('smiles', 'first'),
         log2fc_sp=('log2_fc_estimate', 'median'))
    .reset_index()
    .dropna(subset=['smiles'])
)
sc_novel_props = compute_chem_props(sc_novel['smiles'].tolist())
sc_novel = pd.concat([sc_novel.reset_index(drop=True), sc_novel_props.reset_index(drop=True)], axis=1)
# pec50_pxr / pec50_null are NaN for these

# --- 7c. External ChEMBL rows (not in train)
ext_smiles = chembl_pivot['smiles'].tolist()
ext_props  = compute_chem_props(ext_smiles)
ext = pd.concat([chembl_pivot.reset_index(drop=True), ext_props.reset_index(drop=True)], axis=1)

print(f"Single-conc novel rows: {len(sc_novel):,}")
print(f"External ChEMBL rows:   {len(ext):,}")

In [ ]:
# --- 7d. Stack all sources into one DataFrame
# Define the canonical task columns
NR_TASKS   = [c for c in ['PXR','CAR','VDR','FXR','LXRa','PPARg','PPARa','RXRa']
              if c in chembl_pivot.columns]
TASK_COLS  = ['pec50', 'pec50_null', 'log2fc_sp'] + \
             [f'chembl_{t.lower()}' for t in NR_TASKS] + \
             ['logP', 'MW', 'TPSA']

# Rename ChEMBL pivot columns to match task names
nr_col_map = {t: f'chembl_{t.lower()}' for t in NR_TASKS}
ext = ext.rename(columns=nr_col_map)
# Also map PXR ChEMBL to a separate column (not overwrite pec50)
if 'chembl_pxr' in ext.columns and 'pec50' not in ext.columns:
    ext['pec50'] = np.nan   # primary task NaN for external

# Concatenate all sources
mt_all = pd.concat(
    [core, sc_novel, ext],
    ignore_index=True,
    sort=False
)
# Ensure all task columns exist
for col in TASK_COLS:
    if col not in mt_all.columns:
        mt_all[col] = np.nan

mt_all = mt_all.dropna(subset=['smiles']).reset_index(drop=True)

print(f"\nFull multitask matrix: {mt_all.shape}")
print(f"Task coverage:")
for col in TASK_COLS:
    n = mt_all[col].notna().sum()
    print(f"  {col:20s}: {n:,} ({100*n/len(mt_all):.1f}%)")

# Cache for re-use
mt_all[['smiles', 'inchikey'] + TASK_COLS].to_parquet(
    DATA_PROCESSED / 'multitask_matrix.parquet', index=False
)
print("\nSaved multitask_matrix.parquet")

In [ ]:
# ── 8. Chemprop model helpers ─────────────────────────────────────────────────
N_TASKS = len(TASK_COLS)
print(f"Training with {N_TASKS} tasks: {TASK_COLS}")

def make_dataset(smiles: list[str], y_scaled: np.ndarray) -> cdata.MoleculeDataset:
    return cdata.MoleculeDataset([
        cdata.MoleculeDatapoint.from_smi(smi, y=yi)
        for smi, yi in zip(smiles, y_scaled)
    ])

def make_mpnn(n_tasks: int, depth: int = 3, hidden_dim: int = 300,
              ffn_layers: int = 3, dropout: float = 0.15) -> cmodels.MPNN:
    return cmodels.MPNN(
        message_passing=cnn.BondMessagePassing(depth=depth, d_h=hidden_dim),
        agg=cnn.MeanAggregation(),
        predictor=cnn.RegressionFFN(
            n_tasks=n_tasks, n_layers=ffn_layers, dropout=dropout
        ),
    )

def make_loader(ds, batch_size: int = 64, shuffle: bool = True):
    return cdata.build_dataloader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

print(f"MPNN params: {sum(p.numel() for p in make_mpnn(N_TASKS).parameters()):,}")

In [ ]:
# ── 9. Per-task scaling (must be done per-fold to avoid leakage) ───────────────
def fit_task_scaling(y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Compute per-task mean and std from non-NaN values."""
    means = np.nanmean(y, axis=0)
    stds  = np.nanstd(y, axis=0, ddof=1)
    stds  = np.where(stds < 1e-6, 1.0, stds)
    return means, stds

def scale_targets(y: np.ndarray, means: np.ndarray, stds: np.ndarray) -> np.ndarray:
    return (y - means) / stds   # NaN preserved

def unscale_task0(p_scaled: np.ndarray, means: np.ndarray, stds: np.ndarray) -> np.ndarray:
    return p_scaled[:, 0] * stds[0] + means[0]

print("Scaling helpers defined.")

In [ ]:
# ── 10. 5-fold scaffold CV on primary task (pEC50_pxr) ───────────────────────
# Only the challenge training rows have pec50_pxr labels → CV on those rows only.
# External rows contribute to the loss in every fold (always in train set).

FIXED_EPOCHS = 35
core_idx = mt_all.index[mt_all['pec50'].notna()].tolist()  # primary-task rows
ext_idx  = mt_all.index[mt_all['pec50'].isna()].tolist()   # external-only rows

core_smiles    = mt_all.loc[core_idx, 'smiles'].tolist()
core_y         = mt_all.loc[core_idx, TASK_COLS].values.astype(float)
core_scaffolds = pd.Series(core_smiles).map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(core_scaffolds, n_splits=5, seed=42)

ext_smiles = mt_all.loc[ext_idx, 'smiles'].tolist()
ext_y      = mt_all.loc[ext_idx, TASK_COLS].values.astype(float)

print(f"Primary-task rows (CV): {len(core_idx):,}")
print(f"External rows (always train): {len(ext_idx):,}")

oof_preds = np.full(len(core_idx), np.nan)
fold_metrics = []
t0 = time.time()

for fold, (tr_local, va_local) in enumerate(splits):
    t_fold = time.time()
    print(f"\nFold {fold+1}/5 — {len(tr_local):,} core-train + {len(ext_idx):,} ext / {len(va_local):,} val")

    # Combine core train rows + ALL external rows
    tr_smiles = [core_smiles[i] for i in tr_local] + ext_smiles
    tr_y_raw  = np.vstack([core_y[tr_local], ext_y])

    # Per-fold scaling on TRAINING data only
    means, stds = fit_task_scaling(tr_y_raw)
    tr_y_sc = scale_targets(tr_y_raw, means, stds)
    va_y_sc = scale_targets(core_y[va_local], means, stds)

    ds_tr = make_dataset(tr_smiles, tr_y_sc)
    ds_va = make_dataset([core_smiles[i] for i in va_local], va_y_sc)
    loader_tr = make_loader(ds_tr, batch_size=64,  shuffle=True)
    loader_va = make_loader(ds_va, batch_size=128, shuffle=False)

    mpnn    = make_mpnn(N_TASKS)
    trainer = L.Trainer(
        max_epochs=FIXED_EPOCHS, accelerator='cpu',
        enable_progress_bar=True, enable_model_summary=False, logger=False,
    )
    trainer.fit(mpnn, loader_tr)

    raw   = trainer.predict(mpnn, loader_va)
    p_sc  = torch.cat(raw).numpy()
    p_pxr = unscale_task0(p_sc, means, stds)

    y_true = core_y[va_local, 0]
    m = compute_metrics(y_true, p_pxr)
    m['fold'] = fold
    fold_metrics.append(m)
    oof_preds[va_local] = p_pxr

    elapsed = time.time() - t_fold
    print(f"  RAE={m['RAE']:.4f}  MAE={m['MAE']:.4f}  Spearman={m['Spearman']:.4f}  ({elapsed/60:.1f}m)")

print(f"\nTotal CV time: {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── 11. CV results ────────────────────────────────────────────────────────────
from pxr.eval import rae as rae_fn

y_core = core_y[:, 0]
oof_rae = rae_fn(y_core, oof_preds)

cv_df = pd.DataFrame(fold_metrics)
print("5-fold scaffold CV — expanded multitask Chemprop:")
print(cv_df[['fold','RAE','MAE','Spearman']].to_string(index=False))
print(f"\nMean per-fold RAE : {cv_df['RAE'].mean():.4f} ± {cv_df['RAE'].std():.4f}")
print(f"OOF RAE (global)  : {oof_rae:.4f}")
print()
print("== Comparison ==")
print(f"  Chemprop MT (08, 2-task, cleaned) OOF RAE: 0.5736")
print(f"  Chemprop MT (10, {N_TASKS}-task, + ext)    OOF RAE: {oof_rae:.4f}")

In [ ]:
# ── 12. Train final model on ALL data ─────────────────────────────────────────
print("Training final model on all available data ...")
all_smiles = mt_all['smiles'].tolist()
all_y      = mt_all[TASK_COLS].values.astype(float)

means_full, stds_full = fit_task_scaling(all_y)
all_y_sc = scale_targets(all_y, means_full, stds_full)

ds_full  = make_dataset(all_smiles, all_y_sc)
loader_f = make_loader(ds_full, batch_size=64, shuffle=True)

mpnn_final = make_mpnn(N_TASKS)
trainer_f  = L.Trainer(
    max_epochs=40, accelerator='cpu',
    enable_progress_bar=True, enable_model_summary=False, logger=False,
)
t_train = time.time()
trainer_f.fit(mpnn_final, loader_f)
print(f"Final model training time: {(time.time()-t_train)/60:.1f} min")

In [ ]:
# ── 13. Predict test set ──────────────────────────────────────────────────────
te_dpts   = [cdata.MoleculeDatapoint.from_smi(s) for s in te.smiles]
te_ds     = cdata.MoleculeDataset(te_dpts)
te_loader = make_loader(te_ds, batch_size=128, shuffle=False)

raw_te     = trainer_f.predict(mpnn_final, te_loader)
chemprop10 = torch.cat(raw_te).numpy()[:, 0] * stds_full[0] + means_full[0]
print(f"Chemprop-10 test preds: {chemprop10.min():.2f} – {chemprop10.max():.2f} "
      f"(median {np.median(chemprop10):.3f})")

In [ ]:
# ── 14. Build ensemble: Chemprop-10 + LGBM_aug (inverse-RAE weights) ──────────
# Load LGBM_aug predictions from notebook 09 (or fall back to notebook 07)
try:
    lgbm_sub = pd.read_csv(SUBMISSIONS / '09_lgbm_pipeline.csv')
    lgbm_src = '09'
except FileNotFoundError:
    lgbm_sub = pd.read_csv(SUBMISSIONS / '07_final_ensemble.csv')
    lgbm_src = '07'

lgbm_preds = lgbm_sub.set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
print(f"LGBM preds from submission {lgbm_src}")

# Inverse-RAE weights: use measured OOF RAE for this model vs LGBM baseline
lgbm_rae      = 0.5582   # from notebook 07
chemprop10_rae = oof_rae
w_cp = (1/chemprop10_rae) / (1/chemprop10_rae + 1/lgbm_rae)

final_preds = w_cp * chemprop10 + (1.0 - w_cp) * lgbm_preds
y_min, y_max = core_y[:, 0].min(), core_y[:, 0].max()
final_preds  = np.clip(final_preds, y_min - 0.5, y_max + 0.5)

print(f"w_chemprop = {w_cp:.3f}  |  w_lgbm = {1-w_cp:.3f}")
print(f"Ensemble range: {final_preds.min():.2f} – {final_preds.max():.2f}")

In [ ]:
# ── 15. Save submission ───────────────────────────────────────────────────────
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         final_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all()

out = SUBMISSIONS / '10_expanded_multitask.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"Tasks: {N_TASKS}  |  Chemprop weight: {w_cp:.3f}")
print(sub['pEC50'].describe().round(3))

In [ ]:
# ── 16. Summary plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(y_core, oof_preds, alpha=0.2, s=8, c='steelblue')
lo, hi = y_core.min()-0.2, y_core.max()+0.2
axes[0].plot([lo,hi],[lo,hi],'r--',lw=1.5,alpha=0.7)
axes[0].set(xlabel='True pEC50', ylabel='OOF pred', title=f'OOF (RAE={oof_rae:.4f})')

axes[1].bar(range(1,6), [m['RAE'] for m in fold_metrics], color='steelblue', edgecolor='k', lw=0.5)
axes[1].axhline(oof_rae, color='red', ls='--', lw=1.5, label=f'OOF {oof_rae:.4f}')
axes[1].axhline(0.5736,  color='gray', ls=':', lw=1.5, label='08 (2-task)')
axes[1].set(xlabel='Fold', ylabel='RAE', title='Per-fold RAE')
axes[1].legend(fontsize=9)

axes[2].hist(final_preds, bins=30, color='forestgreen', edgecolor='k', lw=0.4)
axes[2].set(xlabel='pEC50', title=f'Submission 10 (median={np.median(final_preds):.3f})')

plt.tight_layout()
plt.savefig(FIGURES / '10_expanded_multitask.png', bbox_inches='tight')
plt.show()
print("Saved 10_expanded_multitask.png")

## Summary

| Model | Tasks | OOF RAE | Note |
|---|---|---|---|
| Chemprop MT 03 | 2 | 0.5736 (single-fold est.) | pEC50 + null |
| Chemprop MT 08 | 2 | 0.5736 (5-fold) | cleaned train |
| **Chemprop MT 10** | **11** | **see above** | + ext NR + SC + props |

**Key additions that drive improvement**:
1. **Single-conc log2FC** (21k) — largest dataset, direct PXR signal
2. **ChEMBL PXR** (~3k) — additional dose-response pEC50 from literature
3. **Chemical property heads** (logP / MW / TPSA) — dense supervision for every compound,
   regularises the encoder toward physically-meaningful representations
4. **Nuclear receptor relatives** (CAR, VDR, FXR, PPARγ…) — shared binding-pocket
   vocabulary, particularly CAR (NR1I3 — closest relative of PXR)

**Saved**: `submissions/10_expanded_multitask.csv`